# Imports & Setup

In [ ]:
%load_ext autoreload
%autoreload 2

from datetime import date, datetime, timedelta
from functools import partial
from typing import Dict, Optional

import numpy as np
import polars as pl
from tqdm import tqdm
import matplotlib.pyplot as plt

from okx.store import OrderbookStore
from okx.recipes.forwards import build_forwards_pchip, build_forwards_kalman, assign_forwards
from okx.recipes.options import prepare_options
from evaluation.forwards_eval import evaluate_parity, summarize_parity, evaluate_pillar_fit, evaluate_loeo, summarize_pillar_fit, run_full_evaluation

from evaluation.plotting import plot_error_histogram, plot_error_over_time, plot_error_by_group

store = OrderbookStore(
    data_root="data/okx",
    manifest_path="data/okx/manifest.sqlite"
)

# OB Data Fetch, Preprocessing, and Merging

### 1. Define parameters

In [24]:
def make_dates(start_date, n_days):
    dates = [start_date + timedelta(days=i) for i in range(n_days)]
    return dates
binning = '10m'
shared_params = {
    'inst_family': 'BTC-USD',
    'dates': make_dates(date(2025, 9, 1), 30),
    'verbose': True,
    'binning': binning,
    'cache_name': f'arb_check_{binning}',
    'batch_days': 5
}

In [27]:
store.clear_cache()

Cleared all caches


### 2. Fetch futures & options data, preprocess, merge, and calculate moneyness

In [31]:
futures_features = ['trim', 'strip', 'bin_ff', 'sink_bins', 'mid', 'tenor']
futures_lf = store.get(
    inst_type='FUTURES',
    depth=1,
    **shared_params,
    features=futures_features,
).rename({'bid_1_px': 'F_bid', 'ask_1_px': 'F_ask', 'mid': 'F_mid'}).drop('symbol')
print(f"Loaded {futures_lf.select(pl.len()).collect().item()} futures")
print(f"Futures columns: {futures_lf.collect_schema().names()}")
options_features = ['trim', 'strip', 'nullify', 'bin_ff', 'sink_bins', 'drop_nulls_strict', 'parse_option', 'tenor']
options_lf = store.get(
    inst_type='OPTION',
    depth=1,
    **shared_params,
    features=options_features,
)
print(f"Loaded {options_lf.select(pl.len()).collect().item()} options")
print(f"Options columns: {options_lf.collect_schema().names()}")

# Pair options
from okx.recipes.options import _pair_options
options_lf = _pair_options(options_lf)
print(f"Paired options: {options_lf.select(pl.len()).collect().item()}")
print(f"Paired options columns: {options_lf.collect_schema().names()}")

# Join and add moneyness
merged = futures_lf.join(options_lf, on=['timeMs', 'expiry', 'T'], how='inner')
merged = merged.with_columns((pl.col('strike') / pl.col('F_mid')).log().alias('moneyness'))
print(f"Merged: {merged.select(pl.len()).collect().item()}")
print(f"Merged columns: {merged.collect_schema().names()}")


[store] Getting BTC-USD/FUTURES for 30 dates (depth=1, 10m binning, 6 features)


Processing batches:   0%|          | 0/6 [00:00<?, ?it/s]

Loaded 29819 futures
Futures columns: ['timeMs', 'F_bid', 'F_ask', 'F_mid', 'expiry', 'T']
[store] Getting BTC-USD/OPTION for 30 dates (depth=1, 10m binning, 8 features)


Processing batches:   0%|          | 0/6 [00:00<?, ?it/s]

Loaded 2638927 options
Options columns: ['symbol', 'timeMs', 'bid_1_px', 'ask_1_px', 'strike', 'opt_type', 'expiry', 'T']
Paired options: 1211136
Paired options columns: ['timeMs', 'expiry', 'strike', 'T', 'call_bid_1_px', 'call_ask_1_px', 'put_bid_1_px', 'put_ask_1_px']
Merged: 762753
Merged columns: ['timeMs', 'F_bid', 'F_ask', 'F_mid', 'expiry', 'T', 'strike', 'call_bid_1_px', 'call_ask_1_px', 'put_bid_1_px', 'put_ask_1_px', 'moneyness']


### 3. Add BTC-USD column from spot for numeraire conversion 

In [ ]:
spot_lf = store.get(
    inst_type='SPOT',
    depth=0,
    **shared_params,
    features=['trim', 'strip', 'bin_ff', 'sink_bins']
).drop('symbol').rename({'mid': 'BTC-USD'})
print(f"Spot: {spot_lf.select(pl.len()).collect().item()}")
print(f"Spot columns: {spot_lf.collect_schema().names()}")

# Add BTC-USD column
merged = merged.join(spot_lf, on=['timeMs'], how='left')
options_contract_multiplier = 0.1
merged = merged.with_columns((pl.col('BTC-USD') * options_contract_multiplier).alias('conversion')).drop('BTC-USD')

[store] Getting BTC-USD/SPOT for 30 dates (depth=0, 10m binning, 4 features)


Processing batches:   0%|          | 0/6 [00:00<?, ?it/s]

Spot: 4314
Spot columns: ['timeMs', 'BTC-USD']
Merged row count: 762753
Merged row count: 762753


# Synthetic Futures
We will define prices for a number of order types for synthetic futures longs and shorts.

Using put-call parity: **F = K + C - P**

### Synthetic Futures Longs (Buy Call + Sell Put):
- **LONG_LMT_Cb_Pa**: limit order - buy call @ bid, sell put @ ask (best possible fill)
- **LONG_SYM_Cb_Pb**: symmetric - buy call @ bid, sell put @ bid
- **LONG_SYM_Ca_Pa**: symmetric - buy call @ ask, sell put @ ask
- **LONG_MKT_Ca_Pb**: market order - buy call @ ask, sell put @ bid (worst possible fill)

### Synthetic Futures Shorts (Sell Call + Buy Put):
- **SHORT_LMT_Ca_Pb**: limit order - sell call @ ask, buy put @ bid (best possible fill)
- **SHORT_SYM_Ca_Pa**: symmetric - sell call @ ask, buy put @ ask
- **SHORT_SYM_Cb_Pb**: symmetric - sell call @ bid, buy put @ bid
- **SHORT_MKT_Cb_Pa**: market order - sell call @ bid, buy put @ ask (worst possible fill)

### Arbitrage Logic:
- **Long Synthetic**: Profit if synthetic price < futures price → buy synthetic, sell futures
- **Short Synthetic**: Profit if synthetic price > futures price → sell synthetic, buy futures

In [ ]:
# Define synthetic futures order types
# Structure: {name: (formula, description)}
# For LONG synthetic: F_synth = K + C - P (buy call, sell put)
# For SHORT synthetic: F_synth = K - C + P (sell call, buy put)

SYNTHETIC_ORDER_TYPES = {
    # LONG positions (buy call, sell put)
    'LONG_LMT_Cb_Pa': {
        'formula': lambda df: pl.col('strike') + pl.col('call_bid_1_px') - pl.col('put_ask_1_px'),
        'description': 'Long limit: buy call @ bid, sell put @ ask',
        'direction': 'long',
        'execution': 'limit'
    },
    'LONG_SYM_Cb_Pb': {
        'formula': lambda df: pl.col('strike') + pl.col('call_bid_1_px') - pl.col('put_bid_1_px'),
        'description': 'Long symmetric: buy call @ bid, sell put @ bid',
        'direction': 'long',
        'execution': 'symmetric'
    },
    'LONG_SYM_Ca_Pa': {
        'formula': lambda df: pl.col('strike') + pl.col('call_ask_1_px') - pl.col('put_ask_1_px'),
        'description': 'Long symmetric: buy call @ ask, sell put @ ask',
        'direction': 'long',
        'execution': 'symmetric'
    },
    'LONG_MKT_Ca_Pb': {
        'formula': lambda df: pl.col('strike') + pl.col('call_ask_1_px') - pl.col('put_bid_1_px'),
        'description': 'Long market: buy call @ ask, sell put @ bid',
        'direction': 'long',
        'execution': 'market'
    },
    # SHORT positions (sell call, buy put)
    'SHORT_LMT_Ca_Pb': {
        'formula': lambda df: pl.col('strike') - pl.col('call_ask_1_px') + pl.col('put_bid_1_px'),
        'description': 'Short limit: sell call @ ask, buy put @ bid',
        'direction': 'short',
        'execution': 'limit'
    },
    'SHORT_SYM_Ca_Pa': {
        'formula': lambda df: pl.col('strike') - pl.col('call_ask_1_px') + pl.col('put_ask_1_px'),
        'description': 'Short symmetric: sell call @ ask, buy put @ ask',
        'direction': 'short',
        'execution': 'symmetric'
    },
    'SHORT_SYM_Cb_Pb': {
        'formula': lambda df: pl.col('strike') - pl.col('call_bid_1_px') + pl.col('put_bid_1_px'),
        'description': 'Short symmetric: sell call @ bid, buy put @ bid',
        'direction': 'short',
        'execution': 'symmetric'
    },
    'SHORT_MKT_Cb_Pa': {
        'formula': lambda df: pl.col('strike') - pl.col('call_bid_1_px') + pl.col('put_ask_1_px'),
        'description': 'Short market: sell call @ bid, buy put @ ask',
        'direction': 'short',
        'execution': 'market'
    },
}

# Add all synthetic futures columns
synthetic_columns = [
    order_type_def['formula'](merged).alias(order_type_name)
    for order_type_name, order_type_def in SYNTHETIC_ORDER_TYPES.items()
]

merged = merged.with_columns(synthetic_columns)

In [ ]:
# Example: Detect arbitrage opportunities for each order type
# You can now iterate through all synthetic order types to find arbitrage

def detect_arbitrage(df, order_type_name, futures_bid='F_bid', futures_ask='F_ask', threshold=0):
    """
    Detect arbitrage opportunities for a given synthetic order type.
    
    For LONG synthetic: arbitrage when synthetic < futures ask (buy synthetic, sell futures)
    For SHORT synthetic: arbitrage when synthetic > futures bid (sell synthetic, buy futures)
    
    Args:
        df: Polars DataFrame with futures prices and synthetic prices
        order_type_name: Name of synthetic order type column
        futures_bid: Name of futures bid column
        futures_ask: Name of futures ask column
        threshold: Minimum arbitrage profit threshold
    
    Returns:
        DataFrame with arbitrage opportunities and profit columns
    """
    order_type_info = SYNTHETIC_ORDER_TYPES[order_type_name]
    
    if order_type_info['direction'] == 'long':
        # Buy synthetic, sell futures at bid
        profit_col = pl.col(futures_bid) - pl.col(order_type_name)
        arb_condition = profit_col > threshold
    else:  # short
        # Sell synthetic, buy futures at ask
        profit_col = pl.col(order_type_name) - pl.col(futures_ask)
        arb_condition = profit_col > threshold
    
    return df.with_columns([
        profit_col.alias(f'{order_type_name}_profit'),
        arb_condition.alias(f'{order_type_name}_arbitrage')
    ])

# Apply arbitrage detection for all order types
for order_type_name in SYNTHETIC_ORDER_TYPES.keys():
    merged = detect_arbitrage(merged, order_type_name)

print(f"Added arbitrage columns for {len(SYNTHETIC_ORDER_TYPES)} order types")
print(f"Total columns: {len(merged.collect_schema().names())}")


In [ ]:
# Summarize arbitrage opportunities across all order types
def summarize_arbitrage_opportunities(df, order_types_dict):
    """
    Summarize arbitrage statistics for all order types.
    
    Returns a dictionary with statistics for each order type.
    """
    results = {}
    
    for order_type_name, order_type_info in order_types_dict.items():
        profit_col = f'{order_type_name}_profit'
        arb_col = f'{order_type_name}_arbitrage'
        
        stats = df.select([
            pl.col(arb_col).sum().alias('count'),
            pl.col(arb_col).mean().alias('frequency'),
            pl.when(pl.col(arb_col))
              .then(pl.col(profit_col))
              .mean()
              .alias('avg_profit'),
            pl.when(pl.col(arb_col))
              .then(pl.col(profit_col))
              .max()
              .alias('max_profit'),
            pl.col(profit_col).mean().alias('avg_profit_all'),
        ]).collect()
        
        results[order_type_name] = {
            'direction': order_type_info['direction'],
            'execution': order_type_info['execution'],
            'description': order_type_info['description'],
            'count': stats['count'][0],
            'frequency': stats['frequency'][0],
            'avg_profit': stats['avg_profit'][0],
            'max_profit': stats['max_profit'][0],
            'avg_profit_all': stats['avg_profit_all'][0],
        }
    
    return results

# Get summary
arb_summary = summarize_arbitrage_opportunities(merged, SYNTHETIC_ORDER_TYPES)

# Display results
print("\n=== ARBITRAGE OPPORTUNITY SUMMARY ===\n")
for order_type_name, stats in arb_summary.items():
    print(f"{order_type_name}:")
    print(f"  {stats['description']}")
    print(f"  Count: {stats['count']:,} | Frequency: {stats['frequency']:.4%}")
    print(f"  Avg Profit (when arb): ${stats['avg_profit']:.2f}")
    print(f"  Max Profit: ${stats['max_profit']:.2f}")
    print(f"  Avg Profit (all): ${stats['avg_profit_all']:.2f}")
    print()


In [ ]:
# Find best arbitrage opportunity at each timeMs/expiry/strike
# This identifies which order type gives the best profit for each option pair

# Create profit columns list
profit_cols = [f'{order_type}_profit' for order_type in SYNTHETIC_ORDER_TYPES.keys()]

# Find max profit and which order type achieves it
merged_with_best = merged.with_columns([
    pl.max_horizontal(profit_cols).alias('best_profit'),
])

# Create a column indicating which order type is best
def get_best_order_type_expr():
    """Create expression to determine which order type has the highest profit"""
    conditions = []
    for order_type in SYNTHETIC_ORDER_TYPES.keys():
        profit_col = f'{order_type}_profit'
        conditions.append(
            pl.when(pl.col(profit_col) == pl.col('best_profit'))
            .then(pl.lit(order_type))
        )
    
    # Chain all conditions with otherwise clause
    expr = conditions[0]
    for cond in conditions[1:]:
        expr = expr.otherwise(cond)
    return expr.otherwise(pl.lit(None))

merged_with_best = merged_with_best.with_columns([
    get_best_order_type_expr().alias('best_order_type'),
    (pl.col('best_profit') > 0).alias('has_arbitrage')
])

# Summary of best opportunities
best_summary = merged_with_best.select([
    pl.col('has_arbitrage').sum().alias('total_arbitrage_opportunities'),
    pl.col('has_arbitrage').mean().alias('arbitrage_frequency'),
    pl.when(pl.col('has_arbitrage'))
      .then(pl.col('best_profit'))
      .mean()
      .alias('avg_best_profit'),
    pl.col('best_profit').max().alias('max_profit'),
]).collect()

print("\n=== BEST ARBITRAGE OPPORTUNITIES (ACROSS ALL ORDER TYPES) ===")
print(f"Total opportunities: {best_summary['total_arbitrage_opportunities'][0]:,}")
print(f"Frequency: {best_summary['arbitrage_frequency'][0]:.4%}")
print(f"Avg profit (when profitable): ${best_summary['avg_best_profit'][0]:.2f}")
print(f"Max profit: ${best_summary['max_profit'][0]:.2f}")

# Count which order types are best
best_order_type_counts = (
    merged_with_best
    .filter(pl.col('has_arbitrage'))
    .group_by('best_order_type')
    .agg(pl.len().alias('count'))
    .sort('count', descending=True)
    .collect()
)

print("\n=== MOST PROFITABLE ORDER TYPES ===")
for row in best_order_type_counts.iter_rows(named=True):
    order_type = row['best_order_type']
    count = row['count']
    if order_type:
        print(f"{order_type}: {count:,} times ({count/best_summary['total_arbitrage_opportunities'][0]:.2%})")
        print(f"  {SYNTHETIC_ORDER_TYPES[order_type]['description']}")


In [15]:
metrics = (
    merged
    .group_by(['timeMs', 'expiry'])
    .agg([
        pl.len().alias('num_rows'),
        pl.col('moneyness').min().alias('min_moneyness'),
        pl.col('moneyness').max().alias('max_moneyness'),
    ])
)

import numpy as np

# Collect metrics to a DataFrame
metrics_df = metrics.collect().to_pandas()

def print_stats(series, name):
    print(f"{name}: mean={np.mean(series):.3f}, min={np.min(series):.3f}, max={np.max(series):.3f}, "
          f"p1={np.percentile(series, 1):.3f}, p5={np.percentile(series, 5):.3f}, "
          f"p50={np.percentile(series, 50):.3f}, p95={np.percentile(series, 95):.3f}, "
          f"p99={np.percentile(series, 99):.3f}")

print("\nAggregate metrics for joined timeMs/expiry buckets:")
for col in ["num_rows", "min_moneyness", "max_moneyness"]:
    print_stats(metrics_df[col].values, col)



Aggregate metrics for joined timeMs/expiry buckets:
num_rows: mean=25.746, min=3.000, max=42.000, p1=9.000, p5=15.000, p50=26.000, p95=35.000, p99=39.000
min_moneyness: mean=-0.662, min=-1.404, max=0.027, p1=-1.394, p5=-1.377, p50=-0.591, p95=-0.131, p99=-0.061
max_moneyness: mean=0.467, min=0.037, max=1.183, p1=0.085, p5=0.119, p50=0.491, p95=1.025, p99=1.172
